# Federated Evo2-1B LoRA fine-tuning with BioNeMo and NVFlare
This walkthrough prepares the public Nucleotide Transformer splice-site benchmark, creates one common Evo2 LoRA and classification-head initialization, runs sample-weighted FedAvg across three simulated institutions, and evaluates the reloaded global checkpoint.

The default run uses one H100. NVFlare sets `num_threads=1`, and that job's workspace lock serializes its fresh external trainers so only one BioNeMo trainer owns the GPU at a time. Jobs using different workspaces must be run sequentially when they share a GPU.

## Pinned environment
- PyTorch container: `nvcr.io/nvidia/pytorch:26.02-py3`
- BioNeMo Recipes: [`101d88b7e0b0ee40e5af303575eb9661620a3706`](https://github.com/NVIDIA-BioNeMo/bionemo-recipes/tree/101d88b7e0b0ee40e5af303575eb9661620a3706)
- Megatron Bridge v0.5.0: [`fcbb6031103d0ca845c1a54d4fee55ecfcca17b6`](https://github.com/NVIDIA-NeMo/Megatron-Bridge/tree/fcbb6031103d0ca845c1a54d4fee55ecfcca17b6)
- causal-conv1d v1.6.1: [`c51519fb10d92ffa257c1982e1831d3243454b2f`](https://github.com/Dao-AILab/causal-conv1d/tree/c51519fb10d92ffa257c1982e1831d3243454b2f), rebuilt against the container's PyTorch ABI
- Evo2 checkpoint: `evo2/1b-8k-bf16:1.0`
- Dataset revision: `851f9946252e90c665cdb3cc3eedb78f1f26197c`

Build the image and start the container as described in [README.md](./README.md) before running this notebook. The upstream BioNeMo tutorial documents Evo2-1B LoRA on one 48 GB GPU; measure memory and runtime for your exact environment.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

EXAMPLE_DIR = Path.cwd().resolve()
if not (EXAMPLE_DIR / "job.py").is_file():
    EXAMPLE_DIR = Path("/workspace/nvflare/examples/advanced/bionemo/evo2")
assert (EXAMPLE_DIR / "job.py").is_file(), f"Run from the Evo2 example directory: {EXAMPLE_DIR}"
os.chdir(EXAMPLE_DIR)

DATA_DIR = EXAMPLE_DIR / "data"
MODEL_DIR = EXAMPLE_DIR / "models"
RESULTS_DIR = EXAMPLE_DIR / "results"
BASE_CHECKPOINT = MODEL_DIR / "evo2_1b_bf16_mbridge"
INITIAL_CHECKPOINT = MODEL_DIR / "evo2_lora_init.pt"

ACCURACY_MATCHING_PROTOCOL = False
RUN_BASELINES = False
BACKEND = "bionemo"
NUM_CLIENTS = 3
SEQ_LENGTH = 600
LEARNING_RATE = 0.0005
MIN_LEARNING_RATE = 0.00005
SEED = 1234
LORA_DIM = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

if ACCURACY_MATCHING_PROTOCOL:
    RUN_NAME = "fedavg_lora_37x24_gbs96"
    SHOWCASE_SIZE = 0
    NUM_ROUNDS = 37
    LOCAL_STEPS = 24
    TRAIN_MICRO_BATCH_SIZE = 32
    GLOBAL_BATCH_SIZE = 96
    WARMUP_ITERS = 30
    EVAL_ITERS = 1
    PERSIST_CLIENT_TRAINING_STATE = True
else:
    RUN_NAME = "fedavg_lora"
    SHOWCASE_SIZE = 3000
    NUM_ROUNDS = 10
    LOCAL_STEPS = 20
    TRAIN_MICRO_BATCH_SIZE = 4
    GLOBAL_BATCH_SIZE = 32
    WARMUP_ITERS = 2
    EVAL_ITERS = 10
    PERSIST_CLIENT_TRAINING_STATE = False

WORKSPACE = Path(
    "/tmp/nvflare/evo2_splice_fedavg_37x24_gbs96"
    if ACCURACY_MATCHING_PROTOCOL
    else "/tmp/nvflare/evo2_splice_fedavg"
)
TRAINING_STATE_ARGS = ["--persist-client-training-state"] if PERSIST_CLIENT_TRAINING_STATE else []
MIN_ACCURACY = 0.9563
MIN_MACRO_F1 = 0.956
MIN_CLASS_RECALL = 0.94
MIN_CLASS_F1 = 0.94
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Example:", EXAMPLE_DIR)
print("Data:", DATA_DIR)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<not set>"))

## 1. Prepare the public data
The pinned `splice_sites_all` subset has 30,000 training and 3,000 official test sequences of 600 bases. The script reserves 10% of the source training set for validation. The compact configuration selects a stratified 3,000-sequence showcase from the rest; the accuracy-matching configuration keeps the full 27,000-example training split. It audits exact duplicates and overlapping genomic windows before writing three IID site partitions.

No sequence data is bundled with NVFlare. The dataset card says this task comes from [GENCODE V44 human gene annotations](https://www.gencodegenes.org/human/release_44.html). Before downloading, review the pinned [dataset files and card](https://huggingface.co/datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks_revised/tree/851f9946252e90c665cdb3cc3eedb78f1f26197c), the linked [Nucleotide Transformer CC BY-NC-SA 4.0 license](https://github.com/instadeepai/nucleotide-transformer/blob/main/LICENSE.md), and the GENCODE terms. Confirm that those terms govern the exact snapshot and fit your intended use; the pinned card does not declare a standalone SPDX license for the sequence content.

In [ ]:
subprocess.run(
    [
        "python3",
        "prepare_data.py",
        "--output-dir",
        str(DATA_DIR),
        "--num-sites",
        str(NUM_CLIENTS),
        "--partition",
        "iid",
        "--showcase-size",
        str(SHOWCASE_SIZE),
        "--validation-fraction",
        "0.1",
        "--seed",
        "42",
    ],
    check=True,
)

In [ ]:
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(json.dumps({
    "source": manifest["source"],
    "settings": manifest["settings"],
    "counts": manifest["counts"],
    "label_histograms": manifest["label_histograms"],
    "audit": manifest["audit"],
}, indent=2))

Use `--showcase-size 0` for all available training examples. Use `--partition dirichlet --dirichlet-alpha 0.5` to create deterministic label-skew sites.

## 2. Download and convert Evo2-1B
BioNeMo downloads `evo2/1b-8k-bf16:1.0` and converts it once from NeMo 2 to the Megatron Bridge checkpoint layout. All clients reuse these identical frozen base weights and tokenizer settings.

In [ ]:
subprocess.run(
    ["python3", "prepare_base_checkpoint.py", "--output", str(BASE_CHECKPOINT)],
    check=True,
)
print("Megatron Bridge checkpoint:", BASE_CHECKPOINT)

## 3. Export a common initialization
This instantiates the Evo2 classifier once and saves only its rank-16 LoRA and classification-head tensors. BF16 model values are promoted to CPU float32 for initialization, aggregation, checkpoints, and the raw trainable tensor payload. The NVFlare server uses this file as the common initial state for every client.

In [ ]:
subprocess.run(
    [
        "torchrun",
        "--standalone",
        "--nproc_per_node=1",
        "prepare_initial_model.py",
        "--base-checkpoint",
        str(BASE_CHECKPOINT),
        "--data-file",
        str(DATA_DIR / "train" / "pooled.jsonl"),
        "--output",
        str(INITIAL_CHECKPOINT),
        "--work-dir",
        "/tmp/nvflare/evo2_initialize",
        "--seq-length",
        str(SEQ_LENGTH),
        "--seed",
        str(SEED),
        "--peft-mode",
        "lora",
        "--lora-dim",
        str(LORA_DIM),
        "--lora-alpha",
        str(LORA_ALPHA),
        "--lora-dropout",
        str(LORA_DROPOUT),
    ],
    check=True,
)

In [ ]:
def evaluate_checkpoint(
    checkpoint: Path,
    name: str,
    peft_mode: str = "lora",
    reference_report: Path | None = None,
    require_performance_gate: bool = False,
    split_role: str = "test",
) -> Path:
    if split_role not in {"validation", "test"}:
        raise ValueError(f"Unsupported split role: {split_role}")
    split_file = DATA_DIR / ("validation.jsonl" if split_role == "validation" else "test.jsonl")
    result_dir = RESULTS_DIR / name
    report = result_dir / "evaluation.json"
    matrix = result_dir / "confusion_matrix.png"
    command = [
            "torchrun",
            "--standalone",
            "--nproc_per_node=1",
            "evaluate.py",
            "--checkpoint",
            str(checkpoint),
            "--base-checkpoint",
            str(BASE_CHECKPOINT),
            "--test-file",
            str(split_file),
            "--manifest",
            str(DATA_DIR / "manifest.json"),
            "--split-role",
            split_role,
            "--output",
            str(report),
            "--confusion-matrix",
            str(matrix),
            "--work-dir",
            f"/tmp/nvflare/evo2_{name}_evaluation",
            "--seq-length",
            str(SEQ_LENGTH),
            "--micro-batch-size",
            "8",
            "--global-batch-size",
            "32",
            "--seed",
            str(SEED),
            "--peft-mode",
            peft_mode,
            "--lora-dim",
            str(LORA_DIM),
            "--lora-alpha",
            str(LORA_ALPHA),
            "--lora-dropout",
            str(LORA_DROPOUT),
    ]
    if reference_report is not None:
        command.extend([
            "--reference-report", str(reference_report),
            "--require-improvement",
            "--improvement-metric", "macro_f1",
        ])
    if require_performance_gate:
        command.extend([
            "--min-accuracy", str(MIN_ACCURACY),
            "--min-macro-f1", str(MIN_MACRO_F1),
            "--min-class-recall", str(MIN_CLASS_RECALL),
            "--min-class-f1", str(MIN_CLASS_F1),
        ])
    subprocess.run(command, check=True)
    return report

if RUN_BASELINES:
    initial_report = None
    print("Initialization test evaluation is deferred until after fixed-model validation and local selection.")
else:
    initial_report = evaluate_checkpoint(INITIAL_CHECKPOINT, "initialization")
    json.loads(initial_report.read_text())

## 4. Federated LoRA
The compact configuration runs three clients for ten rounds and 20 local optimizer steps per client with microbatch 4 and global batch 32. Set `ACCURACY_MATCHING_PROTOCOL=True` in the configuration cell for the full split and the predeclared 37-round × 24-step, microbatch-32, global-batch-96 persistent-state protocol. The clients send only LoRA and head differences; FedAvg weights each update by its site's training-example count. The baseline commands inherit the same optimizer-state setting.

Before building the data loader, each stateless task advances Megatron's cyclic-sampler cursor by `round × local_steps × global_batch_size`. The pinned cyclic sampler derives each epoch's permutation from that epoch's state; the configured seed still binds common initialization and the training configuration. Fresh processes select the same per-round slices as one uninterrupted deterministic sample stream. The optimizer, scheduler, and runtime RNG state still restart in every stateless task. For continuous site-local optimizer dynamics, add `--persist-client-training-state`. That opt-in mode restores the site's native optimizer, scheduler, RNG, and sampler state, while the global LoRA and head remain the only federated values. The native state stays under the site's workspace, grows with every round, and must start at round zero. Its manifest binds SHA-256 content identities for the base checkpoint, classifier, training JSONL, and validation JSONL, so resumption rejects same-path input edits. A site commits private state before returning its round update; if a later site or the server fails before aggregation, preserve the partial workspace as evidence and restart persistent training from round zero in a fresh workspace.

`--start-round N` instead performs stateless continuation from a global checkpoint produced at round `N - 1`. The checkpoint's continuation signature must match the current mode, clients, sample weights, audited training identities, partition inputs, validation identity, and seed/local-step/batch sampler budget. It cannot be combined with persistent client state because the global checkpoint does not contain any site's private optimizer-state chain.

In [ ]:
federated_command = [
    "python3",
    "job.py",
    "--backend",
    BACKEND,
    "--mode",
    "fedavg",
    "--data-dir",
    str(DATA_DIR),
    "--initial-checkpoint",
    str(INITIAL_CHECKPOINT),
    "--base-checkpoint",
    str(BASE_CHECKPOINT),
    "--workspace",
    str(WORKSPACE),
    "--num-clients",
    str(NUM_CLIENTS),
    "--num-rounds",
    str(NUM_ROUNDS),
    "--start-round",
    "0",
    "--local-steps",
    str(LOCAL_STEPS),
    "--seq-length",
    str(SEQ_LENGTH),
    "--micro-batch-size",
    str(TRAIN_MICRO_BATCH_SIZE),
    "--global-batch-size",
    str(GLOBAL_BATCH_SIZE),
    "--learning-rate",
    str(LEARNING_RATE),
    "--min-learning-rate",
    str(MIN_LEARNING_RATE),
    "--warmup-iters",
    str(WARMUP_ITERS),
    "--eval-iters",
    str(EVAL_ITERS),
    "--seed",
    str(SEED),
    "--peft-mode",
    "lora",
    "--lora-dim",
    str(LORA_DIM),
    "--lora-alpha",
    str(LORA_ALPHA),
    "--lora-dropout",
    str(LORA_DROPOUT),
    "--gpu",
    "[0]",
    "--num-threads",
    "1",
    *TRAINING_STATE_ARGS,
]
subprocess.run(federated_command, check=True)

## 5. Reload and evaluate the global checkpoint
`SimEnv` adds the recipe name below the requested workspace. Evaluation rebuilds the frozen backbone, strictly loads the saved float32 trainable tensors at the BF16 model boundary, and computes accuracy, macro-F1, a classification report, and a confusion matrix on the unchanged official test split. An evaluation-only wrapper adds each zero-based JSONL row index on CPU; the model still receives exactly the pinned input tensors. Evaluation fails unless the sampled indices cover every held-out row exactly once and each label matches its source row. The command also binds `test.jsonl` to `manifest.files.test` and verifies its audited path, row count, byte count, and SHA-256 digest. Evaluation requires paired `--manifest` and `--split-role` arguments by default; use `--allow-unbound-evaluation` only for an intentional JSONL evaluation outside the prepared-data manifest.
For `ACCURACY_MATCHING_PROTOCOL=True`, first evaluate raw rounds 27–36 and their uniform parameter average on validation only. Select by macro-F1, then accuracy, using the earliest raw round to break ties between raw checkpoints. Freeze the selected raw or averaged checkpoint and set `EVO2_SELECTED_CHECKPOINT` to its path before continuing. Only that candidate receives the single official-test evaluation and the acceptance gate; the final round is not selected automatically.

When `RUN_BASELINES=True`, the ordinary initialization/global test calls below are deferred. The baseline section first evaluates all six fixed models on validation, freezes the local selection and checkpoint hashes, and only then evaluates each of those six models on the official test exactly once.


In [ ]:
def global_checkpoint_from_summary(workspace: Path) -> Path:
    summary = json.loads((workspace / "run_summary.json").read_text())
    print(json.dumps({
        key: summary[key]
        for key in (
            "global_checkpoint",
            "global_round_checkpoints",
            "aggregation_weights",
            "total_client_runtime_seconds",
            "peak_client_gpu_memory_mebibytes",
            "total_received_mebibytes",
            "total_sent_mebibytes",
        )
    }, indent=2))
    return Path(summary["global_checkpoint"])

FINAL_GLOBAL_CHECKPOINT = global_checkpoint_from_summary(WORKSPACE)
if RUN_BASELINES:
    GLOBAL_CHECKPOINT = None
    fedavg_report = None
    fedavg_metrics = None
    print("Global official-test evaluation is deferred to the six-model matched-baseline block.")
else:
    if ACCURACY_MATCHING_PROTOCOL:
        selected_checkpoint = os.environ.get("EVO2_SELECTED_CHECKPOINT")
        if not selected_checkpoint:
            raise RuntimeError(
                "Set EVO2_SELECTED_CHECKPOINT to the frozen validation-selected raw or averaged checkpoint."
            )
        GLOBAL_CHECKPOINT = Path(selected_checkpoint).resolve()
    else:
        GLOBAL_CHECKPOINT = FINAL_GLOBAL_CHECKPOINT
    assert GLOBAL_CHECKPOINT.is_file(), GLOBAL_CHECKPOINT
    fedavg_report = evaluate_checkpoint(
        GLOBAL_CHECKPOINT,
        RUN_NAME,
        reference_report=initial_report,
        require_performance_gate=ACCURACY_MATCHING_PROTOCOL,
    )
    fedavg_metrics = json.loads(fedavg_report.read_text())
    print(json.dumps({
        "accuracy": fedavg_metrics["accuracy"],
        "macro_f1": fedavg_metrics["macro_f1"],
        "num_examples": fedavg_metrics["num_examples"],
        "evaluation_coverage": fedavg_metrics["evaluation_coverage"],
        "checkpoint_mebibytes": fedavg_metrics["checkpoint_mebibytes"],
        "runtime_seconds": fedavg_metrics["runtime_seconds"],
        "peak_gpu_memory_mebibytes": fedavg_metrics["peak_gpu_memory_mebibytes"],
        "confusion_matrix": fedavg_metrics["confusion_matrix"],
        "comparison": fedavg_metrics["comparison"],
        "performance_gate": fedavg_metrics.get("performance_gate"),
    }, indent=2))


In [ ]:
from IPython.display import Image, display

if fedavg_metrics is not None:
    display(Image(filename=fedavg_metrics["confusion_matrix_plot"]))
else:
    print("Confusion matrices are displayed after the matched six-model comparison.")

Training succeeds when the reloaded global model improves over the common initialization and the client logs show that adapter and head tensors changed while the backbone stayed frozen. Whether federated LoRA beats local-only, pooled, or head-only training is an experimental result.

## 6. Baselines with documented budgets
The primary local-only comparison is defined against the full 37-round × 24-step H100 FL protocol. Set `ACCURACY_MATCHING_PROTOCOL=True` and rerun data preparation so every local site contains 9,000 examples. Each site-local model starts from the common initialization and runs one uninterrupted task with `--num-rounds 1 --local-steps 888`, microbatch 32, and global batch 96. This is 85,248 presentations per site (`888 × 96`), exactly the work that site contributed to FL (`37 × 24 × 96`). Across the three local runs, 2,664 optimizer steps and 255,744 presentations match the FL system total.

The local commands intentionally omit `--persist-client-training-state`. One process owns all 888 steps, so its optimizer, scheduler, RNG, and sampler remain continuous without a later process restore. The 37-round FL run retains persistent site-private state across its fresh tasks. For a one-contributor round, the aggregator completes its normal validation and clones the client difference directly, preserving the sole FP32 update bit for bit instead of multiplying and dividing by the 9,000-example weight. `--mode local --site-index N` passes only `data/train/site-N.jsonl` to that trainer. The common preflight also hashes every JSONL in the full prepared-data manifest; those shared hashes prove that all runs use the same audited partition snapshot and do not expose another site's sequences to the trainer.

Use fixed endpoints for the primary comparison: raw FL round 36 and the final checkpoint from each of the three local runs. Freeze all four paths and SHA-256 digests before official-test evaluation. The uniform parameter mean of FL rounds 27–36 is a predeclared secondary result. Also rank the three fixed local finals by validation macro-F1, then validation accuracy, then lowest numeric site index, and identify the winner as the validation-selected best local model. Test metrics never select a local model.

Report every site's accuracy, macro-F1, per-class recall and F1, and confusion matrix. Then report the arithmetic mean, population standard deviation, and minimum-to-maximum spread across sites. Because every model sees the same 3,000 official-test rows, the spread is descriptive and the three confusion matrices must not be presented as 9,000 independent examples. Keep the selected local result alongside the all-site results and summary.

The compute-matched pooled run uses `NUM_CLIENTS × NUM_ROUNDS` rounds, matching the federated total number of client optimizer steps and fresh process launches. Federated head-only uses the same three-site budget as federated LoRA. Pooled and federated baselines with multiple fresh tasks retain the target FL run's optimizer-state policy.

For the portable six-model comparison, the code below validates the current run's immutable round checkpoints and constructs the uniform mean of rounds 27 through 36 with FP64 accumulators and FP32 output. It evaluates initialization, raw FL round 36, that predeclared secondary mean, and all three local finals on the manifest-bound validation split; freezes their current checkpoint hashes and the local selection; then evaluates the same six models on the manifest-bound test split and invokes `summarize_baselines.py` with the generated campaign manifest.

The comparison uses evaluation microbatch 8 and global batch 32 for every validation and test endpoint: raw FL round 36, each local final, validation-based local ranking, and the secondary federated average. Portable commands may choose another common microbatch only when it divides every evaluated split exactly; the initialization and every comparison model must share the same microbatch and global batch so their evaluation signatures remain comparable.

The configuration cell leaves `RUN_BASELINES=False` because these runs are expensive. On one H100, run FL and every local, pooled, and head-only job sequentially.


In [ ]:
from collections import OrderedDict
from hashlib import sha256
from statistics import fmean, pstdev

import torch

PRIMARY_LOCAL_NUM_ROUNDS = 1
PRIMARY_LOCAL_STEPS = 888
PRIMARY_TRAIN_MICRO_BATCH_SIZE = 32
PRIMARY_GLOBAL_BATCH_SIZE = 96
PRIMARY_WARMUP_ITERS = 30
PRIMARY_EVAL_ITERS = 1

if RUN_BASELINES and not ACCURACY_MATCHING_PROTOCOL:
    raise RuntimeError(
        "Set ACCURACY_MATCHING_PROTOCOL=True and rerun the notebook from data preparation "
        "before running the fixed-endpoint local comparison."
    )

local_commands = []
for site_index in range(1, NUM_CLIENTS + 1):
    local_commands.append([
        "python3", "job.py",
        "--backend", BACKEND,
        "--mode", "local",
        "--site-index", str(site_index),
        "--data-dir", str(DATA_DIR),
        "--initial-checkpoint", str(INITIAL_CHECKPOINT),
        "--base-checkpoint", str(BASE_CHECKPOINT),
        "--workspace", f"/tmp/nvflare/evo2_splice_local_{site_index}",
        "--require-fresh-workspace",
        "--num-rounds", str(PRIMARY_LOCAL_NUM_ROUNDS),
        "--start-round", "0",
        "--local-steps", str(PRIMARY_LOCAL_STEPS),
        "--seq-length", str(SEQ_LENGTH),
        "--micro-batch-size", str(PRIMARY_TRAIN_MICRO_BATCH_SIZE),
        "--global-batch-size", str(PRIMARY_GLOBAL_BATCH_SIZE),
        "--learning-rate", str(LEARNING_RATE),
        "--min-learning-rate", str(MIN_LEARNING_RATE),
        "--warmup-iters", str(PRIMARY_WARMUP_ITERS),
        "--eval-iters", str(PRIMARY_EVAL_ITERS),
        "--seed", str(SEED),
        "--peft-mode", "lora",
        "--lora-dim", str(LORA_DIM),
        "--lora-alpha", str(LORA_ALPHA),
        "--lora-dropout", str(LORA_DROPOUT),
        "--gpu", "[0]",
        "--num-threads", "1",
    ])

pooled_command = [
    "python3", "job.py",
    "--backend", BACKEND,
    "--mode", "pooled",
    "--data-dir", str(DATA_DIR),
    "--initial-checkpoint", str(INITIAL_CHECKPOINT),
    "--base-checkpoint", str(BASE_CHECKPOINT),
    "--workspace", "/tmp/nvflare/evo2_splice_pooled",
    "--num-rounds", str(NUM_CLIENTS * NUM_ROUNDS),
    "--start-round", "0",
    "--local-steps", str(LOCAL_STEPS),
    "--seq-length", str(SEQ_LENGTH),
    "--micro-batch-size", str(TRAIN_MICRO_BATCH_SIZE),
    "--global-batch-size", str(GLOBAL_BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--min-learning-rate", str(MIN_LEARNING_RATE),
    "--warmup-iters", str(WARMUP_ITERS),
    "--eval-iters", str(EVAL_ITERS),
    "--seed", str(SEED),
    "--peft-mode", "lora",
    "--lora-dim", str(LORA_DIM),
    "--lora-alpha", str(LORA_ALPHA),
    "--lora-dropout", str(LORA_DROPOUT),
    "--gpu", "[0]",
    "--num-threads", "1",
    *TRAINING_STATE_ARGS,
]


def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def comparison_metrics(report: Path) -> dict:
    metrics = json.loads(report.read_text())
    class_metrics = {
        str(class_id): metrics["classification_report"][str(class_id)]
        for class_id in metrics["class_ids"]
    }
    return {
        "checkpoint": metrics["checkpoint"],
        "checkpoint_sha256": metrics["checkpoint_sha256"],
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "class_metrics": class_metrics,
        "min_class_recall": min(row["recall"] for row in class_metrics.values()),
        "min_class_f1": min(row["f1-score"] for row in class_metrics.values()),
        "confusion_matrix": metrics["confusion_matrix"],
    }


if RUN_BASELINES:
    import evo2_adapter_checkpoint as adapter_checkpoint

    local_initial_checkpoint_sha256 = file_sha256(INITIAL_CHECKPOINT)
    fedavg_summary = json.loads((WORKSPACE / "run_summary.json").read_text())
    if fedavg_summary.get("initial_checkpoint_sha256") != local_initial_checkpoint_sha256:
        raise RuntimeError("The federated run did not use the current common initialization checkpoint.")

    # Each subprocess completes before the next one starts so one trainer owns the H100 at a time.
    for command in local_commands:
        subprocess.run(command, check=True)

    local_summaries = {}
    local_checkpoints = {}
    for site_index in range(1, NUM_CLIENTS + 1):
        workspace = Path(f"/tmp/nvflare/evo2_splice_local_{site_index}")
        summary = json.loads((workspace / "run_summary.json").read_text())
        expected_site = f"site-{site_index}"
        if set(summary["training_inputs"]["train_files"]) != {expected_site}:
            raise RuntimeError(f"Local run {site_index} did not isolate training to {expected_site}.")
        if summary["num_rounds"] != 1 or summary["local_steps"] != 888:
            raise RuntimeError(f"Local run {site_index} does not have the fixed 1 × 888-step budget.")
        if summary.get("initial_checkpoint_sha256") != local_initial_checkpoint_sha256:
            raise RuntimeError(f"Local run {site_index} did not use the fixed common initialization.")
        if "persist_client_training_state" in summary["configuration"]:
            raise RuntimeError(f"Local run {site_index} unexpectedly used persistent client state.")
        round_checkpoints = summary.get("global_round_checkpoints")
        if (
            not isinstance(round_checkpoints, list)
            or len(round_checkpoints) != 1
            or round_checkpoints[0].get("round") != 0
        ):
            raise RuntimeError(
                f"Local run {site_index} must contain exactly one immutable round-0 checkpoint entry."
            )
        round_checkpoint = round_checkpoints[0]
        checkpoint = Path(round_checkpoint["path"]).resolve()
        if checkpoint.name != "FL_global_model_round_000.pt":
            raise RuntimeError(f"Local run {site_index} round-0 checkpoint path is not immutable: {checkpoint}.")
        observed_checkpoint_sha256 = file_sha256(checkpoint)
        if round_checkpoint.get("sha256") != observed_checkpoint_sha256:
            raise RuntimeError(
                f"Local run {site_index} round-0 checkpoint SHA-256 does not match its file bytes: "
                f"{round_checkpoint.get('sha256')} != {observed_checkpoint_sha256}."
            )
        local_summaries[site_index] = summary
        local_checkpoints[site_index] = checkpoint

    round_entries = fedavg_summary.get("global_round_checkpoints")
    if not isinstance(round_entries, list):
        raise RuntimeError("The federated summary is missing immutable per-round checkpoints.")
    rounds_by_index = {entry.get("round"): entry for entry in round_entries}
    expected_rounds = set(range(NUM_ROUNDS))
    if len(rounds_by_index) != len(round_entries) or set(rounds_by_index) != expected_rounds:
        raise RuntimeError(
            "Federated round checkpoint coverage is incomplete or duplicated: "
            f"expected={sorted(expected_rounds)}, observed={sorted(rounds_by_index)}."
        )

    def validated_round_checkpoint(round_index: int) -> Path:
        entry = rounds_by_index[round_index]
        checkpoint = Path(entry["path"]).resolve()
        if not checkpoint.is_file():
            raise FileNotFoundError(f"Federated round checkpoint does not exist: {checkpoint}")
        observed_sha256 = file_sha256(checkpoint)
        if entry.get("sha256") != observed_sha256:
            raise RuntimeError(
                f"Federated round {round_index} checkpoint SHA-256 does not match its file bytes: "
                f"{entry.get('sha256')} != {observed_sha256}."
            )
        return checkpoint

    fedavg_final_checkpoint = validated_round_checkpoint(36)
    average_rounds = tuple(range(27, 37))
    average_inputs = [validated_round_checkpoint(round_index) for round_index in average_rounds]
    reference_state = adapter_checkpoint.load_nvflare_checkpoint(average_inputs[0])
    reference_schema = adapter_checkpoint.ValidatedTrainableStateSchema(
        reference_state,
        context="Federated round-27 trainable state",
    )
    averaged_state = OrderedDict(
        (name, torch.zeros_like(tensor, dtype=torch.float64))
        for name, tensor in reference_state.items()
    )
    coefficient = 1.0 / len(average_inputs)
    with torch.no_grad():
        for checkpoint in average_inputs:
            state = adapter_checkpoint.load_nvflare_checkpoint(checkpoint, schema=reference_schema)
            for name, tensor in state.items():
                averaged_state[name].add_(tensor, alpha=coefficient)
    averaged_state = OrderedDict(
        (name, tensor.to(dtype=torch.float32))
        for name, tensor in averaged_state.items()
    )
    reference_schema.validate(averaged_state, context="Federated rounds 27-36 uniform mean")

    selected_mean_checkpoint = RESULTS_DIR / "federated_uniform_mean_rounds_027_036.pt"
    average_metadata = adapter_checkpoint.load_nvflare_checkpoint_metadata(average_inputs[-1])
    average_metadata["checkpoint_averaging"] = {
        "method": "uniform_parameter_mean",
        "input_rounds": list(average_rounds),
        "input_checkpoint_sha256": [file_sha256(checkpoint) for checkpoint in average_inputs],
        "coefficient": coefficient,
        "accumulator_dtype": "float64",
        "output_dtype": "float32",
        "official_test_used": False,
    }
    adapter_checkpoint.save_nvflare_checkpoint(
        averaged_state,
        selected_mean_checkpoint,
        metadata=average_metadata,
    )
    round_trip_mean = adapter_checkpoint.load_nvflare_checkpoint(
        selected_mean_checkpoint,
        schema=reference_schema,
    )
    if any(not torch.equal(averaged_state[name], round_trip_mean[name]) for name in averaged_state):
        raise RuntimeError("The saved federated mean does not round-trip bitwise.")

    fixed_checkpoints = {
        "initialization": INITIAL_CHECKPOINT.resolve(),
        "primary_fl_final": fedavg_final_checkpoint,
        "secondary_selected_fl": selected_mean_checkpoint,
        **{
            f"local_site_{site_index}": checkpoint.resolve()
            for site_index, checkpoint in local_checkpoints.items()
        },
    }
    validation_reports = {
        model_id: evaluate_checkpoint(
            checkpoint,
            f"matched_local_validation_{model_id}",
            split_role="validation",
        )
        for model_id, checkpoint in fixed_checkpoints.items()
    }
    validation_metrics = {
        model_id: json.loads(report.read_text())
        for model_id, report in validation_reports.items()
    }
    selected_local_site = max(
        range(1, NUM_CLIENTS + 1),
        key=lambda site_index: (
            validation_metrics[f"local_site_{site_index}"]["macro_f1"],
            validation_metrics[f"local_site_{site_index}"]["accuracy"],
            -site_index,
        ),
    )

    checkpoint_sha256 = {
        "initialization": file_sha256(fixed_checkpoints["initialization"]),
        "primary_fl_final": file_sha256(fixed_checkpoints["primary_fl_final"]),
        "secondary_selected_fl": file_sha256(fixed_checkpoints["secondary_selected_fl"]),
        "local_sites": {
            str(site_index): file_sha256(fixed_checkpoints[f"local_site_{site_index}"])
            for site_index in range(1, NUM_CLIENTS + 1)
        },
    }
    local_resources = {
        str(site_index): {
            "optimizer_steps": 888,
            "sequence_presentations": 85248,
            "runtime_seconds": local_summaries[site_index]["total_client_runtime_seconds"],
            "peak_gpu_memory_mebibytes": local_summaries[site_index]["peak_client_gpu_memory_mebibytes"],
            "received_mebibytes": local_summaries[site_index]["total_received_mebibytes"],
            "sent_mebibytes": local_summaries[site_index]["total_sent_mebibytes"],
        }
        for site_index in range(1, NUM_CLIENTS + 1)
    }
    campaign_manifest = {
        "format_version": 1,
        "checkpoint_sha256": checkpoint_sha256,
        "protocol": {
            "mode": "three independent local runs",
            "num_rounds": 1,
            "local_steps": 888,
            "micro_batch_size": 32,
            "global_batch_size": 96,
            "seed": SEED,
            "evaluation_micro_batch_size": 8,
            "evaluation_global_batch_size": 32,
            "selection": "validation macro_f1, then accuracy, then lowest site ID",
            "secondary_endpoint": "uniform FP64 parameter mean of federated rounds 27-36",
        },
        "resources": {
            "trainer_process_launches": 3,
            "local_sites": local_resources,
            "total_optimizer_steps": sum(item["optimizer_steps"] for item in local_resources.values()),
            "total_sequence_presentations": sum(
                item["sequence_presentations"] for item in local_resources.values()
            ),
            "total_client_runtime_seconds": sum(item["runtime_seconds"] for item in local_resources.values()),
            "peak_client_gpu_memory_mebibytes": max(
                item["peak_gpu_memory_mebibytes"] for item in local_resources.values()
            ),
            "total_received_mebibytes": sum(item["received_mebibytes"] for item in local_resources.values()),
            "total_sent_mebibytes": sum(item["sent_mebibytes"] for item in local_resources.values()),
        },
    }
    campaign_manifest_path = RESULTS_DIR / "matched_local_baseline_manifest.json"
    campaign_manifest_path.write_text(json.dumps(campaign_manifest, indent=2, sort_keys=True) + "\n")
    fixed_endpoint_manifest = {
        "protocol": "six fixed checkpoints validated before official-test evaluation",
        "checkpoint_sha256": checkpoint_sha256,
        "validation_reports": {
            model_id: {
                "path": str(report.resolve()),
                "sha256": file_sha256(report),
            }
            for model_id, report in validation_reports.items()
        },
        "validation_selected_best_local_site": selected_local_site,
        "local_selection_tie_break": ["macro_f1", "accuracy", "lowest_site_index"],
    }
    endpoint_manifest_path = RESULTS_DIR / "primary_fixed_endpoints.json"
    endpoint_manifest_path.write_text(json.dumps(fixed_endpoint_manifest, indent=2, sort_keys=True) + "\n")

    # Official-test evaluation begins only after the six checkpoint identities and local selection are frozen.
    test_reports = {
        model_id: evaluate_checkpoint(
            checkpoint,
            f"matched_local_test_{model_id}",
            split_role="test",
        )
        for model_id, checkpoint in fixed_checkpoints.items()
    }

    summary_path = RESULTS_DIR / "matched_local_baseline_summary.json"
    summary_command = [
        "python3", "summarize_baselines.py",
        "--campaign-manifest", str(campaign_manifest_path),
        "--primary-fl-validation", str(validation_reports["primary_fl_final"]),
        "--primary-fl-test", str(test_reports["primary_fl_final"]),
        "--secondary-selected-fl-validation", str(validation_reports["secondary_selected_fl"]),
        "--secondary-selected-fl-test", str(test_reports["secondary_selected_fl"]),
        "--initialization-validation", str(validation_reports["initialization"]),
        "--initialization-test", str(test_reports["initialization"]),
        "--output", str(summary_path),
    ]
    for site_index in range(1, NUM_CLIENTS + 1):
        summary_command.extend([
            "--site-validation",
            f"{site_index}={validation_reports[f'local_site_{site_index}']}",
            "--site-test",
            f"{site_index}={test_reports[f'local_site_{site_index}']}",
        ])
    subprocess.run(summary_command, check=True)

    baseline_summary = json.loads(summary_path.read_text())
    detailed_test_metrics = {
        model_id: comparison_metrics(report)
        for model_id, report in test_reports.items()
    }
    local_test_metrics = [
        detailed_test_metrics[f"local_site_{site_index}"]
        for site_index in range(1, NUM_CLIENTS + 1)
    ]
    local_class_summary = {}
    for metric_name in ("min_class_recall", "min_class_f1"):
        values = [metrics[metric_name] for metrics in local_test_metrics]
        local_class_summary[metric_name] = {
            "mean": fmean(values),
            "population_standard_deviation": pstdev(values),
            "minimum": min(values),
            "maximum": max(values),
        }
    details_path = RESULTS_DIR / "matched_local_baseline_details.json"
    details_path.write_text(json.dumps({
        "summary": baseline_summary,
        "official_test": detailed_test_metrics,
        "local_minimum_class_metric_summary": local_class_summary,
    }, indent=2, sort_keys=True) + "\n")
    print(json.dumps({
        "summary": baseline_summary,
        "official_test": detailed_test_metrics,
        "local_minimum_class_metric_summary": local_class_summary,
        "details": str(details_path),
    }, indent=2))

    subprocess.run(pooled_command, check=True)
    pooled_checkpoint = global_checkpoint_from_summary(Path("/tmp/nvflare/evo2_splice_pooled"))
    evaluate_checkpoint(pooled_checkpoint, "pooled_lora")
else:
    print("Primary fixed-endpoint local commands (run sequentially):")
    for command in local_commands:
        print(" ".join(command))
    print("\nCompute-matched pooled command:")
    print(" ".join(pooled_command))


In [ ]:
HEAD_INITIAL_CHECKPOINT = MODEL_DIR / "evo2_head_init.pt"
head_initialization_command = [
    "torchrun", "--standalone", "--nproc_per_node=1", "prepare_initial_model.py",
    "--base-checkpoint", str(BASE_CHECKPOINT),
    "--data-file", str(DATA_DIR / "train" / "pooled.jsonl"),
    "--output", str(HEAD_INITIAL_CHECKPOINT),
    "--work-dir", "/tmp/nvflare/evo2_head_initialize",
    "--seq-length", str(SEQ_LENGTH),
    "--seed", str(SEED),
    "--peft-mode", "head-only",
]
head_fedavg_command = [
    "python3", "job.py",
    "--backend", BACKEND,
    "--mode", "fedavg",
    "--peft-mode", "head-only",
    "--data-dir", str(DATA_DIR),
    "--initial-checkpoint", str(HEAD_INITIAL_CHECKPOINT),
    "--base-checkpoint", str(BASE_CHECKPOINT),
    "--workspace", "/tmp/nvflare/evo2_splice_head_only",
    "--num-clients", str(NUM_CLIENTS),
    "--num-rounds", str(NUM_ROUNDS),
    "--start-round", "0",
    "--local-steps", str(LOCAL_STEPS),
    "--seq-length", str(SEQ_LENGTH),
    "--micro-batch-size", str(TRAIN_MICRO_BATCH_SIZE),
    "--global-batch-size", str(GLOBAL_BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--min-learning-rate", str(MIN_LEARNING_RATE),
    "--warmup-iters", str(WARMUP_ITERS),
    "--eval-iters", str(EVAL_ITERS),
    "--seed", str(SEED),
    "--gpu", "[0]",
    "--num-threads", "1",
    *TRAINING_STATE_ARGS,
]

if RUN_BASELINES:
    subprocess.run(head_initialization_command, check=True)
    head_initial_report = evaluate_checkpoint(
        HEAD_INITIAL_CHECKPOINT, "head_initialization", peft_mode="head-only"
    )
    subprocess.run(head_fedavg_command, check=True)
    head_checkpoint = global_checkpoint_from_summary(Path("/tmp/nvflare/evo2_splice_head_only"))
    evaluate_checkpoint(
        head_checkpoint,
        "fedavg_head_only",
        peft_mode="head-only",
        reference_report=head_initial_report,
    )
else:
    print("Head-only initialization:")
    print(" ".join(head_initialization_command))
    print("\nFederated head-only:")
    print(" ".join(head_fedavg_command))

Freeze the raw-round-36 FL checkpoint and all three local final checkpoints before their official-test evaluations. The notebook first evaluates the local finals on validation and selects the best local by macro-F1, then accuracy, then lowest site index. It then evaluates every fixed endpoint on the official test and prints the all-site metrics, validation-selected local, and arithmetic mean, population standard deviation, and range across local models. Keep each confusion matrix separate because the models reuse the same test rows.

Each workspace's `run_summary.json` records `exchange_dtype=float32`, checkpoint paths and hashes, per-task metrics, aggregate client runtime, maximum GPU memory, and raw trainable tensor-value MiB sent and received. It also records full prepared-manifest identities during common preflight. For local runs, verify that the task's training input is only its selected 9,000-row site file. Report the shared manifest hashes as provenance, not as evidence that a local trainer read the other site files. Payload sizes exclude serialization, protocol, and transport overhead.

Use raw FL round 36 versus all three local finals as the primary comparison. Keep the predeclared uniform parameter mean of FL rounds 27–36 as a secondary result and evaluate it with the same current manifest-bound validation and test settings used for the other five fixed models. Report accuracy, macro-F1, per-class recall and F1, confusion matrices, evaluation runtime, checkpoint size, aggregate client runtime, peak memory, and exchanged bytes. The three local jobs exchange no model between institutions; report that separately from any simulator-internal payload bookkeeping in their run summaries.

The H100 reference table in the README came from a precursor snapshot before the example was ported to the current NVFlare Recipe API and the final metadata and atomic-persistence integrity checks. Those measurements have not been rerun from the exact source in this PR, and the official test split had already been observed during earlier engineering. Treat them as reference reproducibility evidence. For a new notebook run, the generated data manifest, round-checkpoint hashes, fixed-endpoint manifest, and evaluation reports are the authoritative record.


In [ ]:
report_paths = sorted(RESULTS_DIR.glob("*/evaluation.json"))
comparison = []
for report_path in report_paths:
    metrics = json.loads(report_path.read_text())
    comparison.append({
        "run": report_path.parent.name,
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "runtime_seconds": metrics["runtime_seconds"],
        "peak_gpu_memory_mebibytes": metrics["peak_gpu_memory_mebibytes"],
        "checkpoint_mebibytes": metrics["checkpoint_mebibytes"],
    })
comparison

## Execution notes
- Keep `--num-threads 1` and `--gpu '[0]'` for the one-H100 path. The workspace lock serializes trainers only within that job, so run jobs with different workspaces sequentially when they share one GPU.
- `--gpu` is the simulator worker's physical-device assignment among the GPUs visible to that host or container. An outer `CUDA_VISIBLE_DEVICES=2` does not make `--gpu '[0]'` relative to that selection because the simulator worker replaces the child-process setting.
- A default federated run initializes Evo2 30 times. Checkpoint loading and process startup may dominate a short demonstration.
- Add `--persist-client-training-state` only when site-private optimizer, scheduler, RNG, and sampler continuity is part of the experiment; budget local disk for one native checkpoint per site and round.
- Use `--start-round N` only for stateless continuation from a round `N - 1` global checkpoint with a matching continuation signature, and do not combine it with persistent client state.
- Before the full run, use two clients, two rounds, and two local steps to validate the environment.
- Lower the microbatch if memory is tight, while keeping the global batch divisible by it.
- The example covers single-GPU LoRA or head-only classification. Full-model federation, long-context tuning, model parallelism, and multi-node execution are later extensions.